# IMPORT LIBRARY

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

# CONNECTING TO DATABASE

In [ ]:
load_dotenv("../.env")

conn = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connection created")

# CUSTOMER BEHAVIOR ANALYSIS

##
RETRIVE DATA

In [ ]:
import pandas as pd

customer_data = '''
    WITH 
    cust_activity AS (
        SELECT
            cm.customer_id
            ,COUNT(i.interaction_id) AS total_interaction
            ,AVG(cm.engagement_score) AS engagement_score
        FROM customers_monthly_metrics cm
        LEFT JOIN interactions i ON cm.customer_id = i.customer_id
        WHERE DATE_PART('year',event_ts)<=2024
        GROUP BY cm.customer_id
    ),
                
    cust_trans AS (
        SELECT
            t.customer_id
            ,p.property_type
            ,t.payment_mode
            ,t.deal_price AS total_deal_price
            ,t.transaction_id
            ,t.deal_date 
        FROM transactions t
        LEFT JOIN properties p ON t.property_id = p.property_id
        WHERE EXTRACT(YEAR FROM t.deal_date) <= 2024 AND t.deal_status = 'Completed'
    ),
    
    cust_property_interaction AS (
        SELECT
            i.customer_id
            ,i.property_id
            ,p.property_type AS cust_property_int
            ,i.event_ts
            ,i.interaction_type
        FROM interactions i
        LEFT JOIN properties p ON p.property_id = i.property_id
        WHERE EXTRACT(YEAR FROM i.event_ts) <= 2024
    ),

    cust_int AS (
        SELECT
            t.customer_id
            ,t.deal_date
            ,i.event_ts
        FROM cust_trans t
        LEFT JOIN interactions i ON t.customer_id = i.customer_id
        WHERE i.event_ts <= t.deal_date
    ),

    cust_day AS (
        SELECT 
             customer_id
            ,deal_date
            ,MAX(event_ts) AS Last_interaction
        FROM cust_int
        GROUP BY customer_id, deal_date
    ),

    first_date AS(
        SELECT t.customer_id
                ,t.deal_date
                ,LAG(t.deal_date,1,c.signup_date)OVER(PARTITION BY t.customer_id ORDER BY t.deal_date) AS first_date
        FROM transactions t
        LEFT JOIN customers c ON t.customer_id = c.customer_id
        WHERE t.deal_status = 'Completed' 
            AND EXTRACT(YEAR FROM t.deal_date)<=2024
    ),

    interaction_before_deal AS(
        SELECT fd.customer_id,
                COUNT(i.interaction_id) AS num_interaction
        FROM first_date fd
        LEFT JOIN interactions i
            ON fd.customer_id = i.customer_id
            AND i.event_ts > fd.first_date
            AND i.event_ts <= fd.deal_date
        GROUP BY fd.customer_id
)
    SELECT 
         c.customer_id
        ,c.age
        ,c.income_band
        ,c.segment
        ,c.household_size
        ,c.home_city
        ,c.signup_date
        ,ca.total_interaction
        ,ca.engagement_score
        ,c.acquisition_channel
        ,ct.transaction_id
        ,ct.deal_date
        ,cp.event_ts
        ,cp.interaction_type
        ,ic.num_interaction
        ,ct.payment_mode
        ,ct.total_deal_price
        ,ct.property_type
        ,cp.cust_property_int
    FROM customers c
    LEFT JOIN cust_trans ct ON ct.customer_id = c.customer_id
    LEFT JOIN cust_activity ca ON ca.customer_id = c.customer_id
    LEFT JOIN cust_property_interaction cp ON cp.customer_id = c.customer_id
    LEFT JOIN cust_day cd ON cd.customer_id = c.customer_id AND cd.deal_date = ct.deal_date
    LEFT JOIN interaction_before_deal ic ON ic.customer_id = c.customer_id
'''

df_customer = pd.read_sql(customer_data, conn)
print(len(df_customer))


In [ ]:
df_customer = df_customer.sort_values('customer_id').reset_index(drop= True)
df_customer['signup_date'] = pd.to_datetime(df_customer['signup_date'])
df_customer['deal_date'] = pd.to_datetime(df_customer['deal_date'])
df_trans_cust = df_customer[df_customer['transaction_id'].notna()]
df_trans_cust = (\
    df_trans_cust
    .drop_duplicates('transaction_id')
    .drop(columns='cust_property_int')
    .sort_values('customer_id')
    .reset_index(drop = True)
)
df_non_trans_cust = df_customer[df_customer['transaction_id'].isnull()].reset_index(drop=True)
df_cust = (\
    df_customer
    .drop_duplicates('customer_id')
    .drop(columns = ['total_interaction',
                     'transaction_id', 'payment_mode',
                     'total_deal_price', 'property_type',
                     'cust_property_int']).sort_values('customer_id').reset_index(drop = True)
)

In [ ]:
def call_crosstab (df,col1,col2,col3):
   df_crosstab = round(
                        pd.crosstab(
                                    df[col1]
                                    ,df[col2]
                                    ,values=df[col3]
                                    ,aggfunc='nunique'
                                    ,margins= True
                                    ,margins_name='Total'
                                    ,normalize=True).fillna(0)*100,2)
   total_row = df_crosstab.loc[['Total']]
   df_segments = df_crosstab.drop('Total')
   df_segments_sorted = df_segments.sort_values('Total',ascending=False)
   df_final = pd.concat([df_segments_sorted,total_row])
   return(df_final)

##
CUSTOMER OVERVIEW

In [ ]:
print(f"Number of Customer : {len(df_cust)}")
print(f"Number of Customer Transaction : {df_trans_cust['customer_id'].nunique()}")
print(f"AVG Revenue per Customer : {(df_trans_cust['total_deal_price'].sum())/(df_trans_cust['customer_id'].nunique())/1e6:,.2f} M")
print("\nProperty Interest by Customer's Household Size")
display(call_crosstab(df_customer,'cust_property_int','household_size','customer_id'))
print("\nProperty Sold by Customer's Household Size")
display(call_crosstab(df_trans_cust,'property_type','household_size','customer_id'))

##
CUSTOMER SEGMENT ANALYSIS

###
1) Segment Distribution

In [ ]:
fig, axes = plt.subplots(4,1 ,figsize=(10,20))

sns.boxplot(
    data= df_cust
    ,x = 'segment'
    ,y = 'age'
    ,ax = axes[0]
)
axes[0].set_title('Age Distrbution by Each Segment')
axes[0].set_ylabel('age')
axes[0].set_xlabel('segment')

sns.boxplot(
    data= df_trans_cust
    ,x = 'segment'
    ,y = 'total_deal_price'
    ,ax = axes[1]
)

axes[1].set_title('Total Deal Price Distribution by Each Segment')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Total Deal Price')

sns.boxplot(
    data= df_cust
    ,x = 'segment'
    ,y = 'household_size'
    ,ax = axes[2]
)

axes[2].set_title('Total Household Size Distribution by Each Segment')
axes[2].set_xlabel('Segment')
axes[2].set_ylabel('Household Size')

sns.boxplot(
    data= df_customer
    ,x = 'segment'
    ,y = 'total_interaction'
    ,ax = axes[3]
)

axes[3].set_title('Total Interaction by Each Segment')
axes[3].set_xlabel('Segment')
axes[3].set_ylabel('Total Interaction')


plt.tight_layout()
plt.show()

###
2) Customer Behavior by Segment

In [ ]:
cust_by_city = (\
    df_cust
    .groupby('home_city')
    .agg(total_customer = ('customer_id','count'))
)

cust_by_city['customer_share'] = (\
    round(cust_by_city['total_customer']/
          (cust_by_city['total_customer'].sum())*100,2)
)

segment_by_city = call_crosstab(df_cust,'home_city','segment','customer_id')
segment_by_property_interest = call_crosstab(df_customer,'cust_property_int','segment','customer_id')
segment_by_income = call_crosstab(df_cust,'income_band','segment','customer_id')
segment_by_interaction_type = call_crosstab(df_customer,'interaction_type','segment','customer_id')
segment_by_acquisition_channel = call_crosstab(df_customer,'acquisition_channel','segment','customer_id')
print(f"{'-'*14}Customer by City{'-'*13}")
display(cust_by_city)
print(f"{'-'*19}% Customer Segment Share by City{'-'*19}")
display(segment_by_city)
print(f"{'-'*15}% Customer Segment Share by Interest Property{'-'*15}")
display(segment_by_property_interest)
print(f"{'-'*19}% Customer Segment Share by Income{'-'*19}")
display(segment_by_income)
print(f"{'-'*15}% Customer Segment Share by Interction Type{'-'*15}")
display(segment_by_interaction_type)
print(f"{'-'*15}% Customer Segment Share by Acquisition Channel{'-'*15}")
display(segment_by_acquisition_channel)


###
3) Customer Transaction Analysis by Segment

In [ ]:

df_segment_perform = (\
    df_cust
    .groupby('segment')
    .agg(total_customer = ('customer_id','count'))
    )

df_segment_perform['customer_share'] = (
    round(df_segment_perform['total_customer']/
    (df_segment_perform['total_customer']
     .sum())*100,2)
    )

df_segment_perform['cust_trans'] = (\
    df_trans_cust
    .groupby('segment')
    ['customer_id'].nunique()
)

df_segment_perform['convertion_rate'] = (\
    round(df_segment_perform['cust_trans']/
    df_segment_perform['total_customer']
    *100,2)
)

df_segment_perform['total_earn'] = (\
    df_trans_cust
    .groupby('segment')
    ['total_deal_price'].sum())

df_segment_perform['revenue_share'] = (\
    round(df_segment_perform['total_earn']/
          (df_segment_perform['total_earn'].sum())
          *100,2)
)

df_segment_perform['avg_per_cust'] = (\
    df_trans_cust
    .groupby('segment')
    ['total_deal_price'].mean()
    )
df_segment_perform = df_segment_perform.sort_values('total_earn',ascending = False).reset_index()


In [ ]:
df_trans = df_trans_cust[df_trans_cust['deal_date']>df_trans_cust['signup_date']].copy()
df_trans['time_to_deal'] = (df_customer['deal_date']-df_customer['signup_date']).dt.days
cust_days = df_trans[df_trans['time_to_deal'].notna()]
cust_days_seg = (
    cust_days.groupby('segment')
    .agg(AVG_days_to_deal = ('time_to_deal','mean'),
         AVG_interaction_to_deal= ('num_interaction','mean'))
).fillna(0).astype(int)
cust_days_seg['AVG_days_to_deal']= cust_days_seg['AVG_days_to_deal'].round(0)
cust_days_seg['AVG_interaction_to_deal']= cust_days_seg['AVG_interaction_to_deal'].round(0)
cust_days_seg = cust_days_seg.sort_values(['AVG_days_to_deal','AVG_interaction_to_deal'])
display(cust_days_seg)

In [ ]:
seg_trans_by_city = call_crosstab(df_trans_cust,'home_city','segment','customer_id')
seg_trans_by_property_type = call_crosstab(df_trans_cust,'property_type','segment','customer_id')
seg_trans_by_income = call_crosstab(df_trans_cust,'income_band','segment','customer_id')
seg_trans_by_interaction_type = call_crosstab(df_trans_cust,'interaction_type','segment','customer_id')
seg_trans_by_acquisition_channel = call_crosstab(df_trans_cust,'acquisition_channel','segment','customer_id')

##OUTPUT
print(f'{"-"*35}Transaction Performance by Customer Segment{"-"*35}')
display(df_segment_perform)
print(f"{'-'*19}% Customer Transaction by City{'-'*19}")
display(seg_trans_by_city)
print(f"{'-'*15}% Customer Transaction by Property Type{'-'*15}")
display(seg_trans_by_property_type)
print(f"{'-'*12}% Customer Transaction by Interest Property{'-'*12}")
display(seg_trans_by_income)
print(f"{'-'*14}% Customer Transaction by Interction Type{'-'*15}")
display(seg_trans_by_interaction_type)
print(f"{'-'*15}% Customer Transaction by Acquisition Channel{'-'*15}")
display(seg_trans_by_acquisition_channel)

###
4) Customer Distribution by Deal Price Tier and Segment

In [ ]:
def cross_tab (df,col1,col2,col3):
   df_crosstab = pd.crosstab(df[col1]
                            ,df[col2]
                            ,values=df[col3]
                            ,aggfunc='nunique'
                            ,margins= True
                            ,margins_name='Total').fillna(0).astype(int)
   total_row = df_crosstab.loc[['Total']]
   df_segments = df_crosstab.drop('Total')
   df_segments_sorted = df_segments.sort_values('Total',ascending=False)
   df_final = pd.concat([df_segments_sorted,total_row])
   return(df_final)

In [ ]:
premium_cust = df_trans_cust.copy()
bins = [premium_cust['total_deal_price'].min(),
        premium_cust['total_deal_price'].quantile(0.70),
        premium_cust['total_deal_price'].quantile(0.90),
        premium_cust['total_deal_price'].max()]
labels = ['Standard', 'High-Tier', 'Ultra-Luxury']
premium_cust['customer_tier'] = pd.cut(premium_cust['total_deal_price'],bins=bins,labels=labels,include_lowest=True)
display(cross_tab(premium_cust,'segment','customer_tier','customer_id'))

###
5) Customer Distribution by Transaction Frequency and Segment

In [ ]:
df_multiple_trans_cust = df_trans_cust.copy()
df_multiple_trans_cust['num_trans'] = df_multiple_trans_cust.groupby('customer_id')['transaction_id'].transform('count')
display(call_crosstab(df_multiple_trans_cust,'segment','num_trans','customer_id'))

##
CUSTOMER INCOME ANALYSIS

###
1) Customer Distribution by Customer's Income

In [ ]:
fig, axes = plt.subplots(4,1 ,figsize=(10,20))
custom_order = ['Low', 'Lower-Mid', 'Mid', 'Upper-Mid', 'High', 'Unknown']
sns.boxplot(
    data= df_cust
    ,x = 'income_band'
    ,y = 'age'
    ,order = custom_order
    ,ax = axes[0]
)
axes[0].set_title('Age Distrbution by Customer Income')
axes[0].set_ylabel('age')
axes[0].set_xlabel('Income Band')

sns.boxplot(
    data= df_trans_cust
    ,x = 'income_band'
    ,y = 'total_deal_price'
    ,order = custom_order
    ,ax = axes[1]
)

axes[1].set_title('Total Deal Price Distribution by Customer Income')
axes[1].set_xlabel('Income Band')
axes[1].set_ylabel('Total Deal Price')

sns.boxplot(
    data= df_cust
    ,x = 'income_band'
    ,y = 'household_size'
    ,order=custom_order
    ,ax = axes[2]
)

axes[2].set_title('Total Household Size Distribution by Customer Income')
axes[2].set_xlabel('income_band')
axes[2].set_ylabel('Household Size')

sns.boxplot(
    data= df_customer
    ,x = 'income_band'
    ,y = 'total_interaction'
    ,order=custom_order
    ,ax = axes[3]
)

axes[3].set_title('Total Interaction by Customer Income')
axes[3].set_xlabel('Income Band')
axes[3].set_ylabel('Total Interaction')


plt.tight_layout()
plt.show()

In [ ]:
income_by_city = call_crosstab(df_cust,'home_city','income_band','customer_id')
income_by_property_interest = call_crosstab(df_customer,'cust_property_int','income_band','customer_id')
income_by_interaction_type = call_crosstab(df_customer,'interaction_type','income_band','customer_id')
income_by_acquisition_channel = call_crosstab(df_cust,'acquisition_channel','income_band','customer_id')

print(f"{'-'*21}% Customer Income Share by City{'-'*21}")
display(income_by_city)
print(f"{'-'*16}% Customer Income Share by Interest Property{'-'*16}")
display(income_by_property_interest)
print(f"{'-'*16}% Customer Income Share by Interaction Type{'-'*16}")
display(income_by_interaction_type)
print(f"{'-'*16}% Customer Income Share by Acquisition Channel{'-'*16}")
display(income_by_acquisition_channel)

###
2) Customer Transaction Analysis by Customer's Income

In [ ]:
df_cust_income_perform = (\
    df_cust
    .groupby('income_band')
    .agg(total_customer = ('customer_id','count'))
    )

df_cust_income_perform['customer_share'] = (
    round(df_cust_income_perform['total_customer']/
    (df_cust_income_perform['total_customer']
     .sum())*100,2)
    )

df_cust_income_perform['cust_trans'] = (\
    df_trans_cust
    .groupby('income_band')
    ['customer_id'].nunique()
)

df_cust_income_perform['convertion_rate'] = (\
    round(df_cust_income_perform['cust_trans']/
    df_cust_income_perform['total_customer']
    *100,2)
)

df_cust_income_perform['total_earn'] = (\
    df_trans_cust
    .groupby('income_band')
    ['total_deal_price'].sum())

df_cust_income_perform['revenue_share'] = (\
    round(df_cust_income_perform['total_earn']/
          (df_cust_income_perform['total_earn'].sum())
          *100,2)
)

df_cust_income_perform['avg_per_cust'] = (\
    df_trans_cust
    .groupby('income_band')
    ['total_deal_price'].mean()
    )
df_cust_income_perform = df_cust_income_perform.sort_values('total_earn',ascending=False).reset_index()

In [ ]:
df_trans = df_trans_cust[df_trans_cust['deal_date']>df_trans_cust['signup_date']].copy()
df_trans['time_to_deal'] = (df_customer['deal_date']-df_customer['signup_date']).dt.days
cust_days = df_trans[df_trans['time_to_deal'].notna()]
cust_days_inc = (
    cust_days.groupby('income_band')
    .agg(AVG_days_to_deal = ('time_to_deal','mean'),
         AVG_interaction_to_deal= ('num_interaction','mean'))
).fillna(0).astype(int)
cust_days_inc['AVG_days_to_deal']= cust_days_inc['AVG_days_to_deal'].round(0)
cust_days_inc['AVG_interaction_to_deal']= cust_days_inc['AVG_interaction_to_deal'].round(0)
cust_days_inc= cust_days_inc.sort_values(['AVG_days_to_deal','AVG_interaction_to_deal'])

In [ ]:
cust_inc_by_payment_mode = call_crosstab(df_trans_cust,'income_band','payment_mode','customer_id')
cust_inc_by_property_trans = call_crosstab(df_trans_cust,'income_band','property_type','customer_id')

#OUTPUT
print(f'{"-"*36}Transaction Performance by Customer Income{"-"*36}')
display(df_cust_income_perform)
display(cust_days_inc)
print(f'{"-"*23}Type of Property{"-"*20}')
display(cust_inc_by_property_trans)
print(f'{"-"*20}Type of Payment Method{"-"*19}')
display(cust_inc_by_payment_mode)

###
3) Customer Distribution by Deal Price Tier and Customer's Income

In [ ]:
display(cross_tab(premium_cust,'income_band','customer_tier','customer_id'))

###
4) Customer Distribution by Transaction Frequency and Customer's Income

In [ ]:
display(cross_tab(df_multiple_trans_cust,'income_band','num_trans','customer_id'))

##
CUSTOMER WITHOUT TRANSACTION ANALYSIS

In [ ]:
display(cross_tab(df_non_trans_cust,'segment','income_band','customer_id'))
display(df_non_trans_cust.groupby('cust_property_int')['customer_id'].nunique())

##
CUSTOMERS DATA WITHOUT TRANSACTION AND INTERACTION 

In [ ]:
df_non_trans_cust[df_non_trans_cust['cust_property_int'].isna()]